# ДЗ3: function calling

Полное SFT Qwen2.5-0.5B-Instruct для пяти функций умного дома.

## Окружение

Пакеты указаны в requirements.txt. Нужна GPU с поддержкой BF16.

In [ ]:
import gc
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import Dataset
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import SFTConfig, SFTTrainer

from chat_ui import launch_chat
from smart_home import BASELINE_PROMPTS, SMART_HOME_TOOLS, SYSTEM_PROMPT, SmartHome

SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = Path("outputs/full_sft")
MAX_LENGTH = 2048

set_seed(SEED)
assert torch.cuda.is_available(), "Нужен GPU kernel"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 200)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16).to("cuda")

## 1. Генерация

Схемы функций передаём через tools; chat template задаёт формат вызова.

In [ ]:
print(json.dumps(SMART_HOME_TOOLS[0], ensure_ascii=False, indent=2))
print("=" * 80)
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Turn on the kitchen light."},
]
print(tokenizer.apply_chat_template(messages, tools=SMART_HOME_TOOLS, tokenize=False, add_generation_prompt=True))

Жадное декодирование; стоп на `<|im_end|>`. Возвращаем только продолжение.

In [ ]:
@torch.inference_mode()
def generate(model, tokenizer, messages, tools, max_new_tokens=256):
    model.eval()
    inputs = tokenizer.apply_chat_template(
        messages, tools=tools, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    )
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    output = model.generate(
        **inputs, do_sample=False, max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def answer(model, tokenizer, prompt):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
    return generate(model, tokenizer, messages, SMART_HOME_TOOLS)

Десять одинаковых запросов до и после SFT; ответы сохраняем без ручных исправлений.

In [ ]:
comparison = pd.DataFrame({"prompt": BASELINE_PROMPTS})
comparison["base"] = [answer(model, tokenizer, prompt) for prompt in BASELINE_PROMPTS]
display(comparison)

## 2. Данные

Одна строка JSONL содержит диалог и схемы доступных функций.

In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


def show_dialog(messages):
    for message in messages:
        print(f"[{message['role']}] {message.get('content', '')}")
        for call in message.get("tool_calls") or []:
            arguments = json.dumps(call["function"]["arguments"], ensure_ascii=False)
            print(f"    -> {call['function']['name']}({arguments})")


train_rows = read_jsonl("data/train.jsonl")
validation_rows = read_jsonl("data/validation.jsonl")
print(f"train: {len(train_rows)}, validation: {len(validation_rows)}\n")

example = next(row for row in train_rows if any(m.get("tool_calls") for m in row["messages"]))
show_dialog(example["messages"])

Считаем одиночные, параллельные, многошаговые диалоги и ответы без вызова.

In [ ]:
def scenario_type(row):
    calls_per_message = [len(m.get("tool_calls") or []) for m in row["messages"]]
    if sum(calls_per_message) == 0:
        return "no_call"
    if sum(n > 0 for n in calls_per_message) > 1:
        return "multi_turn"
    return "parallel_calls" if max(calls_per_message) > 1 else "single_call"


pd.DataFrame({
    "train": pd.Series([scenario_type(row) for row in train_rows]).value_counts(),
    "validation": pd.Series([scenario_type(row) for row in validation_rows]).value_counts(),
})

Рендерим тот же пример через нативный шаблон.

In [ ]:
print(tokenizer.apply_chat_template(example["messages"], tools=example["tools"], tokenize=False))

### Почему tool response внутри user?

Результат приходит извне, поэтому это вход. Иначе модель училась бы придумывать результаты функций вместо чтения реальных ответов.

### Чему учат четыре типа диалогов?

| Тип | Навык; что потеряем без него |
|---|---|
| single_call | Выбор одной функции и аргументов; без этих примеров хуже покрываются простые команды |
| parallel_calls | Несколько независимых действий за ход; без них нет прямых примеров выполнения составной команды |
| multi_turn | Следующий вызов учитывает предыдущий результат; без таких диалогов пропадают примеры зависимых шагов |
| no_call | Уточнение, отказ или обычный ответ; без этих примеров модель не учится обходиться без вызовов |

## 3. Маска loss

Маска включает ответы assistant, вызовы и `<|im_end|>`; остальные токены получают label `-100`.

In [ ]:
NATIVE_TEMPLATE = tokenizer.chat_template
print(NATIVE_TEMPLATE)

Добавляем generation-теги в нативный шаблон. Проверяем равенство текста на всех train и validation диалогах.

In [ ]:
plain_message = r"{{- '<|im_start|>' + message.role + '\n' + message.content + '<|im_end|>' + '\n' }}"
masked_message = r"""{%- if message.role == "assistant" %}
            {{- '<|im_start|>assistant\n' }}
            {%- generation %}{{- message.content + '<|im_end|>' + '\n' }}{% endgeneration %}
        {%- else %}
            """ + plain_message + """
        {%- endif %}"""
tool_start = "{{- '<|im_start|>' + message.role }}"
tool_end = "{{- '<|im_end|>\\n' }}\n    {%- elif message.role == \"tool\" %}"
assert NATIVE_TEMPLATE.count(plain_message) == 1
assert NATIVE_TEMPLATE.count(tool_start) == 1
assert NATIVE_TEMPLATE.count(tool_end) == 1
TRAIN_TEMPLATE = NATIVE_TEMPLATE.replace(plain_message, masked_message)
TRAIN_TEMPLATE = TRAIN_TEMPLATE.replace(tool_start, tool_start + "{% generation %}")
TRAIN_TEMPLATE = TRAIN_TEMPLATE.replace(tool_end, "{{- '<|im_end|>\\n' }}{% endgeneration %}\n    {%- elif message.role == \"tool\" %}")
tokenizer.chat_template = TRAIN_TEMPLATE

In [ ]:
for row in train_rows + validation_rows:
    arguments = {"tools": row["tools"], "tokenize": False}
    native = tokenizer.apply_chat_template(row["messages"], chat_template=NATIVE_TEMPLATE, **arguments)
    rendered = tokenizer.apply_chat_template(row["messages"], **arguments)
    assert rendered == native, row["id"]
    encoded = tokenizer.apply_chat_template(
        row["messages"], tools=row["tools"], tokenize=True,
        return_dict=True, return_assistant_tokens_mask=True,
    )
    assert len(encoded["input_ids"]) <= MAX_LENGTH, row["id"]
    trained = tokenizer.decode([token for token, flag in zip(encoded["input_ids"], encoded["assistant_masks"]) if flag])
    assert "<|im_end|>" in trained, row["id"]
    assert "<tools>" not in trained and "<tool_response>" not in trained, row["id"]
    if any(message.get("tool_calls") for message in row["messages"]):
        assert "<tool_call>" in trained, row["id"]
print(f"Проверено диалогов: {len(train_rows) + len(validation_rows)}")
encoded = tokenizer.apply_chat_template(
    example["messages"], tools=example["tools"], tokenize=True,
    return_dict=True, return_assistant_tokens_mask=True,
)
trained = tokenizer.decode([token for token, flag in zip(encoded["input_ids"], encoded["assistant_masks"]) if flag])
print(f"Учим {sum(encoded['assistant_masks'])} из {len(encoded['input_ids'])} токенов:\n{trained}")

## 4. Обучение

Обучаем все веса три эпохи. Настройки фиксируем заранее; тестовые запросы для выбора параметров не используем.

In [ ]:
def drop_nulls(value):
    if isinstance(value, dict):
        return {key: drop_nulls(item) for key, item in value.items() if item is not None}
    if isinstance(value, list):
        return [drop_nulls(item) for item in value]
    return value


def to_dataset(rows):
                                                                                                
                                                                                           
    dataset = Dataset.from_list([{"messages": row["messages"], "tools": row["tools"]} for row in rows])
    return dataset.with_transform(lambda batch: {key: [drop_nulls(item) for item in items] for key, items in batch.items()})


train_dataset = to_dataset(train_rows)
validation_dataset = to_dataset(validation_rows)

BF16, batch 2 × accumulation 8, learning rate 2e-5, seed 42; loss только по assistant.

In [ ]:
config = SFTConfig(
    output_dir=str(OUTPUT_DIR), num_train_epochs=3,
    per_device_train_batch_size=2, gradient_accumulation_steps=8,
    per_device_eval_batch_size=2, learning_rate=2e-5,
    lr_scheduler_type="linear", warmup_ratio=0.1,
    bf16=True, max_length=MAX_LENGTH, assistant_only_loss=True,
    logging_steps=5, eval_strategy="epoch", save_strategy="no",
    seed=SEED, data_seed=SEED, report_to="none",
)
trainer = SFTTrainer(
    model=model, args=config, train_dataset=train_dataset,
    eval_dataset=validation_dataset, processing_class=tokenizer,
)
assert all(parameter.requires_grad for parameter in model.parameters())
train_result = trainer.train()
final_eval = trainer.evaluate()
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)
history = pd.DataFrame(trainer.state.log_history)
history.to_json("outputs/training_history.json", orient="records", indent=2)
Path("outputs/training_summary.json").write_text(json.dumps({
    "train": train_result.metrics, "validation": final_eval,
    "parameters": model.num_parameters(), "epochs": config.num_train_epochs,
    "optimizer_steps": trainer.state.global_step, "seed": SEED,
}, ensure_ascii=False, indent=2))

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
train_log = history.dropna(subset=["loss"])
eval_log = history.dropna(subset=["eval_loss"])
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(train_log["step"], train_log["loss"], label="train", color="#386f8f")
ax.plot(eval_log["step"], eval_log["eval_loss"], marker="o", label="validation", color="#bd7242")
ax.set(xlabel="Шаг оптимизатора", ylabel="Loss")
ax.grid(alpha=0.2)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig("outputs/loss.png", dpi=180)
plt.show()

### Как читать loss?

Train loss усредняется между логами, validation loss считается на отложенных диалогах; сравниваем динамику и отдельно проверяем качество вызовов.

## 5. Тест

60 исходных случаев: 40 EN, 20 RU. Метрики шаблона: набор функций и точное совпадение аргументов; сломанный JSON считается ошибкой.

In [ ]:
TOOL_CALL_PATTERN = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.DOTALL)


def parse_tool_calls(text):

    blocks = TOOL_CALL_PATTERN.findall(text)
    if len(blocks) != text.count("<tool_call>"):
        return None
    try:
        calls = [json.loads(block) for block in blocks]
    except ValueError:
        return None
    if not all(isinstance(call, dict) and isinstance(call.get("name"), str) and isinstance(call.get("arguments"), dict) for call in calls):
        return None
    return [{"name": call["name"], "arguments": call["arguments"]} for call in calls]


ARGUMENT_DEFAULTS = {
    tool["function"]["name"]: {name: spec["default"] for name, spec in tool["function"]["parameters"]["properties"].items() if "default" in spec}
    for tool in SMART_HOME_TOOLS
}


def canonical(calls):

    filled = [{"name": c["name"], "arguments": {**ARGUMENT_DEFAULTS.get(c["name"], {}), **c["arguments"]}} for c in calls]
    return sorted(json.dumps(c, sort_keys=True) for c in filled)


def evaluate(model, tokenizer, cases):
    rows = []
    for case in cases:
        messages = case["messages"]
        if messages[0]["role"] != "system":
            messages = [{"role": "system", "content": SYSTEM_PROMPT}, *messages]
        output = generate(model, tokenizer, messages, SMART_HOME_TOOLS)
        calls = parse_tool_calls(output)
        tools_ok = calls is not None and {call["name"] for call in calls} == set(case["expected_tools"])
        rows.append({
            "id": case["id"], "lang": case["lang"], "output": output,
            "tools_ok": tools_ok,
            "args_ok": tools_ok and canonical(calls) == canonical(case["expected_calls"]),
        })
    return pd.DataFrame(rows)


def summarize(results):
    return results.groupby("lang").agg(tool_set_accuracy=("tools_ok", "mean"), exact_match=("args_ok", "mean"))

In [ ]:
del trainer
gc.collect()
torch.cuda.empty_cache()
model.eval()
cases = read_jsonl("data/test_cases.jsonl")
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16).to("cuda")
base_results = evaluate(base_model, tokenizer, cases)
sft_results = evaluate(model, tokenizer, cases)
scores = pd.concat([
    summarize(base_results).assign(model="base"),
    summarize(sft_results).assign(model="sft"),
]).reset_index()[["model", "lang", "tool_set_accuracy", "exact_match"]]
base_results.to_json("outputs/base_predictions.json", orient="records", force_ascii=False, indent=2)
sft_results.to_json("outputs/sft_predictions.json", orient="records", force_ascii=False, indent=2)
del base_model
gc.collect()
torch.cuda.empty_cache()
scores.to_csv("outputs/eval_results.csv", index=False)
display(scores)

Те же десять запросов после обучения.

In [ ]:
comparison["after_sft"] = [answer(model, tokenizer, prompt) for prompt in BASELINE_PROMPTS]
display(comparison)
comparison.to_json("outputs/comparison.json", orient="records", force_ascii=False, indent=2)

## 6. Агент

Каждый вызов выполняем в симуляторе, ответ добавляем как tool и продолжаем генерацию. Останавливаемся после текстового ответа или четырёх шагов.

In [ ]:
def run_agent(model, tokenizer, home, messages, max_steps=4):
    for _ in range(max_steps):
        output = generate(model, tokenizer, messages, SMART_HOME_TOOLS)
        calls = parse_tool_calls(output)
        content = TOOL_CALL_PATTERN.sub("", output).strip() if calls else output
        reply = {"role": "assistant", "content": content}
        if calls:
            reply["tool_calls"] = [
                {"type": "function", "function": call} for call in calls
            ]
        messages.append(reply)
        if not calls:
            break
        for call in calls:
            result = home.call(call["name"], call["arguments"])
            messages.append({
                "role": "tool", "name": call["name"],
                "content": json.dumps(result, ensure_ascii=False),
            })

In [ ]:
scenarios = [
    ["Turn on the kitchen light."],
    ["Turn on the kitchen light and set it to 23 degrees."],
    ["Make it two degrees warmer in the bedroom.", "Two more degrees."],
    ["Thanks!"],
    ["Open the garage."],
]
agent_results = []
for prompts in scenarios:
    home = SmartHome()
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for prompt in prompts:
        messages.append({"role": "user", "content": prompt})
        run_agent(model, tokenizer, home, messages)
    print("=" * 80)
    show_dialog(messages[1:])
    print(f"дом: {json.dumps(home.snapshot(), ensure_ascii=False)}")
    agent_results.append({"prompts": prompts, "messages": messages, "state": home.snapshot()})
Path("outputs/agent_scenarios.json").write_text(json.dumps(agent_results, ensure_ascii=False, indent=2))

Чат использует тот же агентный цикл и показывает текущее состояние дома.

In [ ]:
launch_chat(model, tokenizer, SmartHome(), run_agent)